In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog datatocurnch_novacart_adb")

In [0]:
spark.sql("create schema if not exists bronze_schema")

In [0]:
spark.sql(
"""
create table if not exists datatocurnch_novacart_adb.bronze_schema.ingestion_control (
  layer string,
  table_name string,
  ts_col string,
  pk_col string,
  last_successful_ts timestamp,
  last_successful_pk bigint,
  last_run_id string,
  rows_written bigint,
  run_status string,
  updated_at timestamp
)
using delta
""")

In [0]:
table_config={
    "orders":{"pk_col":"order_id","ts_col":"updated_at"},
    "products":{"pk_col":"product_id","ts_col":"updated_at"},
    "payments":{"pk_col":"payment_id","ts_col":"processed_at"},
}

In [0]:
bronze_run_id=str(uuid.uuid4())
print("Current Bronze Run ID: ",bronze_run_id)

In [0]:
def get_last_successful_watermark(table_name:str):
    ctrl=(
        spark.table("datatocurnch_novacart_adb.bronze_schema.ingestion_control")
        .filter(
            (F.col("table_name")==table_name)&
            (F.col("layer")=="bronze") &
            (F.col("run_status")=="success")
    ).orderBy(F.col("updated_at").desc()).limit(1)
    )

    rows=ctrl.collect()
    if not rows:
        return (None,None)
    return rows[0]["last_successful_ts"],rows[0]["last_successful_pk"]

    

In [0]:
def upsert_bronze_control(
    table_name,
    ts_col,
    pk_col,
    last_ts,
    last_pk,
    rows_written,
    run_id
):

    control_df = spark.createDataFrame(
        [
            (
                "bronze",
                table_name,
                ts_col,
                pk_col,
                last_ts,
                int(last_pk) if last_pk is not None else None,
                run_id,
                int(rows_written),
                "success",
                datetime.utcnow()
            )
        ],
        schema="""
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successful_ts timestamp,
        last_successful_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(
        spark,
        "datatocurnch_novacart_adb.bronze_schema.ingestion_control"
    )

    (
        dt.alias("t")
        .merge(
            control_df.alias("s"),
            "t.table_name = s.table_name AND t.layer = s.layer"
        )
        .whenMatchedUpdate(
            set={
                "ts_col": "s.ts_col",
                "pk_col": "s.pk_col",
                "last_successful_ts": "s.last_successful_ts",
                "last_successful_pk": "s.last_successful_pk",
                "last_run_id": "s.last_run_id",
                "rows_written": "s.rows_written",
                "run_status": "s.run_status",
                "updated_at": "s.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
	

In [0]:
for table_name, cfg in table_config.items():

    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]

    source_table = f"datatocrunch_novacart_sql_connection_catalog.dbo.{table_name}"
    target_table = f"datatocurnch_novacart_adb.bronze_schema.{table_name}_raw"

    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)

    if last_successful_ts is not None:
        last_successful_ts = last_successful_ts.replace(
            microsecond=(last_successful_ts.microsecond // 1000 * 1000)
        )
    print(f"Last Success TS: {last_successful_ts}")
    print(f"Last Success PK: {last_successful_pk}")

    source_df = (
        spark.read.table(source_table)
        .withColumn(ts_col, F.date_trunc("MILLISECOND", F.col(ts_col).cast("timestamp")))
    )

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        if last_successful_pk is None:
            rows_to_load = source_df.filter(
                F.col(ts_col) > F.lit(last_successful_ts)
            )
        else:
            rows_to_load = source_df.filter(
                (F.col(ts_col) > F.lit(last_successful_ts)) |
                (
                    (F.col(ts_col) == F.lit(last_successful_ts)) &
                    (F.col(pk_col).cast("long") > F.lit(last_successful_pk))
                )
            )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )

    rows_count = rows_to_load.count()

    if rows_count == 0:
        print(f"No new rows to load for {table_name}")
        upsert_bronze_control(
            table_name, ts_col, pk_col,
            last_successful_ts, last_successful_pk,
            rows_count, bronze_run_id
        )
        continue

    rows_to_load.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table)

  #  max_ts = rows_to_load.agg(
  #      F.max(ts_col).alias("max_ts")
  #  ).collect()[0]["max_ts"]

  #  max_pk = rows_to_load.filter(
  #      F.col(ts_col) == max_ts
  #  ).agg(
  #      F.max(F.col(pk_col).cast("long")).alias("max_pk")
  #  ).collect()[0]["max_pk"]

    watermark_now = (
        rows_to_load
        .select(ts_col, pk_col)
        .orderBy(
            F.col(ts_col).desc(),
            F.col(pk_col).desc()
        )
        .limit(1)
        .collect()[0]
    )
    max_ts=watermark_now[ts_col]
    max_pk=int(watermark_now[pk_col]) if watermark_now[pk_col] is not None else None
    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        rows_count,
        bronze_run_id
    )
    print(f"Wrote {rows_count} rows to {target_table}")

In [0]:
print("Orders Bronze Count:",spark.sql("select count(*) from datatocurnch_novacart_adb.bronze_schema.orders_raw").collect()[0][0])
print("Products Bronze Count:",spark.sql("select count(*) from datatocurnch_novacart_adb.bronze_schema.products_raw").collect()[0][0])
print("Payments Bronze Count:",spark.sql("select count(*) from datatocurnch_novacart_adb.bronze_schema.payments_raw").collect()[0][0])

display(spark.sql("select * from datatocurnch_novacart_adb.bronze_schema.ingestion_control").orderBy("table_name"))